# Extra development conditions on Kaggle T4

Same Tesla T4. Do not switch GPUs. Do not run Stage C.

This adds only two missing development cells on the same 20 examples:
1. `no_relay_filler` — B sees the question, choices, and dummy document; no cache, no inventory.
2. `hobf_sender_only_no_filler` — OBF on A's cache, sender-only, no dummy document.

The previous 140 rows are merged from `results_stage_b/evaluations.jsonl` if that file is in the uploaded repo. The run writes 40 new receiver evaluations (plus 20 sender passes for OBF).

Download `/kaggle/working/results_stage_b_extra` during the run. Keep `data/test.jsonl` unused.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_b.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_b.py").is_file() and (cand / "data" / "dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_b.py and data/dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_b.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
prior = REPO / "results_stage_b" / "evaluations.jsonl"
print("REPO", REPO.resolve())
print("prior", prior, "exists", prior.is_file())
if not prior.is_file():
    raise FileNotFoundError("Include results_stage_b/evaluations.jsonl in the uploaded zip so the 140 rows are merged.")

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Download /kaggle/working/results_stage_b_extra during the run.")

In [ ]:
cmd = [
    sys.executable, "-m", "src.stage_b", "--split", "dev",
    "--only", "no_relay_filler,hobf_sender_only_no_filler",
    "--prior", str(prior),
]
rc = subprocess.call(cmd)
print("stage_b extra exit", rc)

In [ ]:
src = WORK / "results_stage_b_extra"
if not (src / "summary.json").is_file():
    latest = REPO / "results" / "stage_b" / "latest"
    src = latest
print("reading", src)
summary = json.loads((src / "summary.json").read_text(encoding="utf-8"))
gates = json.loads((src / "gates.json").read_text(encoding="utf-8"))
manifest = json.loads((src / "manifest.json").read_text(encoding="utf-8"))
print("model_revision", manifest.get("model_revision"), manifest.get("extra", {}).get("model_revision"))
print("no_relay_filler", summary.get("no_relay_filler"))
print("hobf_sender_only_no_filler", summary.get("hobf_sender_only_no_filler"))
print("majority_class_rate", summary.get("dev_label_balance"))
print("extra_controls_complete", gates.get("extra_controls_complete"))
print("Do not open data/test.jsonl yet.")